# 04 — Robustness Appendix

Sensitivity of the main specification to choices that theory does not pin down:
how treatment is dated, which years enter the sample, how the outcome is
transformed, how countries are weighted, and which countries are included.

Checks that could invalidate the design rather than qualify a result are in
notebook 01, not here. The distinction matters: notebook 01 establishes that a
single control country carries substantial leverage, which is a property of the
design; this notebook reports what the estimate becomes when that country is
excluded, which is a sensitivity analysis of an already-defined specification.

Nothing in this notebook is a headline result, and no conclusion in the thesis
rests on any single row.

**Input** `data/gbv_panel_analysis.csv`.
**Output** `outputs/tables/`, and a final check of every exported table.

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))

from analysis_helpers import *          # noqa: F401,F403
import estimators as E
import scm as SCM

DATA_PATH = resolve_data_path(PROJECT_DIR)
OUT = make_output_dirs(PROJECT_DIR)
_d = load_and_prepare_data(DATA_PATH, OUT)
df, SAMPLES = _d["df"], _d["SAMPLES"]
country_info, inventory = _d["country_info"], _d["inventory"]
ALL_CTRL = _d["ALL_CTRL"]

## 1. How treatment is dated

The main specification switches treatment on in the calendar year of
ratification. Three alternatives are reported: entry into force, which follows
ratification by three to six months; and, for each of these, the first *full*
calendar year afterwards.

The last pair matters because the panel is annual while the treaty dates are
days. Austria ratified on 14 November 2013 and Bosnia and Herzegovina on
7 November 2013, so coding 2013 as fully treated counts roughly ninety per cent
of an untreated year as treated.

In [ ]:
# Treatment-definition robustness.
#
# The first-full-year variants exist only because the exact treaty dates are
# recorded in data/treatment_dates_verified.csv. Several countries
# ratified in November (Austria 14 Nov 2013, Bosnia 7 Nov 2013, Serbia
# 21 Nov 2013), so coding the calendar year of ratification as fully treated
# counts ~92% of an untreated year as treated.
rows = []
DEFS = [("did_interaction",         "Ratification year (main)"),
        ("did_eif",                 "Entry-into-force year"),
        ("did_first_full_year",     "First full year after ratification"),
        ("did_first_full_year_eif", "First full year after entry into force")]
for k in ["fhr", "male_hom", "log_ratio"]:
    for var, lbl in DEFS:
        if var not in SAMPLES[k].columns:
            print(f"  SKIP {k} {lbl}: {var} not in sample")
            continue
        r = E.feols(SAMPLES[k], k, [var], ["country","year"], cluster="country",
                    name=f"{k} {lbl}")
        row = E.coef_row(r, var, f"{OUTCOME_BY_KEY[k]['label']} - {lbl}")
        row["definition"] = lbl
        row["sample"] = "main"
        rows.append(row)

# Turkey's 2021 withdrawal is the panel's only treatment reversal.
# The main samples KEEP Turkey but drop its three post-withdrawal years
# (2021-2023), since after the withdrawal took effect Turkey is not a
# party and those rows are not treated in any meaningful sense.
# (The panel columns and the build_sample keyword use the treaty's own
# legal term, denunciation, for the same event.)
# To test the alternative -- keep those years and code them as untreated
# -- the three rows are restored and the two codings compared on that
# same sample.
# SAMPLE_WINDOWS is used explicitly so the variant spans exactly the same
# years as SAMPLES[k]; without it build_sample would start in 1990.
for k in ["fhr", "male_hom", "log_ratio"]:
    y0, y1 = SAMPLE_WINDOWS[k]
    s_tk = build_sample(df, k, excl_turkey_post_denunciation=False,
                          year_min=y0, year_max=y1)
    for var, lbl in [("did_interaction",  "TR 2021-23 kept, coded treated"),
                     ("did_nonabsorbing", "TR 2021-23 kept, coded untreated")]:
        if var not in s_tk.columns:
            print(f"  SKIP {k} {lbl}: {var} not in sample")
            continue
        r = E.feols(s_tk, k, [var], ["country","year"], cluster="country",
                    name=f"{k} {lbl}")
        row = E.coef_row(r, var, f"{OUTCOME_BY_KEY[k]['label']} - {lbl}")
        row["definition"] = lbl
        row["sample"] = "Turkey post-withdrawal years restored"
        rows.append(row)

t = export_table(rows, OUT["tables"] / "robustness_treatment_definition.csv",
                 "treatment definition")
print(t[["label","b","se","p","N"]].to_string(index=False))
print()
n_main = len(SAMPLES["fhr"])
y0, y1 = SAMPLE_WINDOWS["fhr"]
n_tk = len(build_sample(df, "fhr", excl_turkey_post_denunciation=False,
                        year_min=y0, year_max=y1))
print(f"Main fhr sample N={n_main}; restoring Turkey 2021-2023 gives N={n_tk} "
      f"(+{n_tk - n_main} rows).")
print("The last two rows per outcome differ ONLY in how those three years are")
print("coded, so the gap between them is the withdrawal's contribution and")
print("nothing else. The untreated coding is valid for TWFE only: stacked DiD")
print("and Callaway-Sant'Anna both require treatment never to switch off.")


## 2. Estimation window and outcome transformation

Homicide rates are bounded below by zero and right-skewed. The main
specification uses the level, which keeps the coefficient in the units of the
outcome; log and inverse-hyperbolic-sine transformations are reported alongside,
together with alternative start years.

In [ ]:
rows = []
for y0 in [1990, 2000, 2005, 2010]:
    s = SAMPLES["fhr"][SAMPLES["fhr"].year >= y0]
    if s.did_interaction.nunique() < 2:
        continue
    r = E.feols(s, "fhr", ["did_interaction"], ["country","year"], cluster="country",
                name=f"window {y0}")
    rows.append(E.coef_row(r, "did_interaction", f"Female homicide, {y0}-2023"))
s = SAMPLES["fhr"].copy()
s["log1p_fhr"] = np.log1p(s.fhr)
r = E.feols(s, "log1p_fhr", ["did_interaction"], ["country","year"], cluster="country",
            name="log1p")
rows.append(E.coef_row(r, "did_interaction", "Female homicide, log(1+rate)"))
t = export_table(rows, OUT["tables"] / "robustness_window_and_transformation.csv", "window/transform")
print(t[["label","b","se","p","N"]].to_string(index=False))

## 3. Weighting and influential countries

Unweighted regression treats San Marino and Germany as equally informative about
the average effect of ratification. Population weighting changes the estimand to
a population-average effect. Both are reported, along with the effect of
excluding each treated country in turn.

In [ ]:
rows = []
s = SAMPLES["fhr"].dropna(subset=["population_total"])
if len(s):
    r = E.feols(s, "fhr", ["did_interaction"], ["country","year"], cluster="country",
                weights="population_total", name="pop weighted")
    rows.append(E.coef_row(r, "did_interaction", "Population-weighted (different estimand)"))
for thr in [500_000, 1_000_000]:
    pop = SAMPLES["fhr"].groupby("country").population_total.mean()
    small = pop[pop < thr].index.tolist()
    sub = SAMPLES["fhr"][~SAMPLES["fhr"].country.isin(small)]
    r = E.feols(sub, "fhr", ["did_interaction"], ["country","year"], cluster="country",
                name=f"excl<{thr}")
    row = E.coef_row(r, "did_interaction", f"Excluding population < {thr:,} ({len(small)} countries)")
    rows.append(row)
t = export_table(rows, OUT["tables"] / "robustness_weighting.csv", "weighting")
print(t[["label","b","se","p","N"]].to_string(index=False))

In [ ]:
# Leave-one-TREATED-country-out
rows = []
for c in sorted(SAMPLES["fhr"].loc[SAMPLES["fhr"].treated_ever==1, "country"].unique()):
    sub = SAMPLES["fhr"][SAMPLES["fhr"].country != c]
    try:
        r = E.feols(sub, "fhr", ["did_interaction"], ["country","year"],
                    cluster="country", name=f"excl {c}")
        rows.append(E.coef_row(r, "did_interaction", c))
    except RuntimeError:
        continue
lot = export_table(rows, OUT["tables"] / "robustness_leave_one_treated_out.csv", "LOO treated")
lot = lot.sort_values("b")
print(f"range: {lot.b.min():.4f} ({lot.label.iloc[0]}) to "
      f"{lot.b.max():.4f} ({lot.label.iloc[-1]})")
print(lot[["label","b","se","p"]].head(3).to_string(index=False))
print(lot[["label","b","se","p"]].tail(3).to_string(index=False))

## 4. Balanced event-time window

Event-time coefficients far from treatment are supported by a shrinking and
non-random subset of countries, because early ratifiers are observed for longer.
This restricts the sample to countries observed across the whole window, so that
composition does not change with event time.

In [ ]:
d = SAMPLES["fhr"].copy()
d["_et"] = np.where(d.treated_ever==1, d.year - d.convention_ratified_year, np.nan)
full = (d[d.treated_ever==1].groupby("country")
        .apply(lambda g: set(range(-6,9)).issubset(set(g._et.dropna().astype(int))),
               include_groups=False))
bal_countries = full[full].index.tolist()
print(f"countries with full -6..+8 coverage: {len(bal_countries)} of "
      f"{d.loc[d.treated_ever==1,'country'].nunique()} treated")
sub = d[(d.treated_ever==0) | d.country.isin(bal_countries)]
r = E.feols(sub, "fhr", ["did_interaction"], ["country","year"], cluster="country",
            name="balanced")
row = E.coef_row(r, "did_interaction", "Balanced -6..+8 window")
export_table([row], OUT["tables"] / "robustness_balanced_event_window.csv", "balanced window")
print(row)

## 5. Consolidated robustness table

`outputs/tables/robustness_results.csv` gathers every sensitivity estimate in
this notebook into one table, so that a claim about robustness in the thesis can
be checked against a single file.

In [ ]:
import glob

# Collect the individual sensitivity tables written above. The consolidated
# file is named robustness_results.csv and would otherwise match this glob on a
# re-run, so it is excluded by name rather than by pattern.
CONSOLIDATED = OUT["tables"] / "robustness_results.csv"

rob = []
for f in sorted(glob.glob(str(OUT["tables"] / "robustness_*.csv"))):
    if Path(f).name == CONSOLIDATED.name:
        continue
    t = pd.read_csv(f)
    t.insert(0, "check", Path(f).stem.replace("robustness_", "").replace("_", " "))
    rob.append(t)

robustness = pd.concat(rob, ignore_index=True)
cols = [c for c in ["check", "label", "b", "se", "p", "ci_low", "ci_high",
                    "N", "clusters"] if c in robustness.columns]
robustness = robustness[cols]
robustness.to_csv(CONSOLIDATED, index=False)
print(f"{len(robustness)} sensitivity estimates across "
      f"{robustness['check'].nunique()} checks")
print(robustness.round(4).to_string(index=False))

## 5. Validity of every exported table

A final pass over every table written by any notebook, confirming that no
standard error is non-finite or non-positive. A cluster-robust variance matrix
that is not positive definite produces such a value silently, so the check is
made explicitly rather than assumed.

The software manifest records the package versions used, for the methods
appendix.

In [ ]:
import glob

# Read back every table any notebook wrote and confirm that no reported standard
# error is non-finite or non-positive. A cluster-robust variance matrix that is
# not positive definite produces such a value without raising, so the check is
# made on the exported files rather than trusted at estimation time.
REPORT = OUT["diagnostics"] / "standard_error_check.csv"
files = sorted(set(glob.glob(str(OUT["tables"] / "*.csv")) +
                   glob.glob(str(OUT["diagnostics"] / "*.csv"))))

bad = []
for f in files:
    if Path(f).name == REPORT.name:
        continue
    try:
        t = pd.read_csv(f)
    except Exception:
        continue
    for col in ["se", "std_error", "Std. Error"]:
        if col in t.columns:
            v = pd.to_numeric(t[col], errors="coerce")
            n = int((~np.isfinite(v)).sum() + (v < 0).sum())
            if n:
                bad.append({"file": Path(f).name, "column": col, "n_bad": n,
                            "n_rows": len(t)})

se_check = pd.DataFrame(bad)
se_check.to_csv(REPORT, index=False)
print(f"Checked {len(files) - 1} exported tables.")
if len(se_check):
    print("NON-FINITE OR NON-POSITIVE STANDARD ERRORS FOUND:")
    print(se_check.to_string(index=False))
else:
    print("No non-finite or non-positive standard errors in any exported table.")
software_manifest(OUT, DATA_PATH)
print("\nRobustness appendix complete.")